# 02 · Reading & writing data

Real data lives in files (or databases / cloud storage). This lesson covers CSV, JSON and **Parquet**,
schemas, bad records, and how Spark writes output.

### The course dataset (in `data/`) — an online shop
| File | Rows | Columns |
|---|---|---|
| `customers.csv` | 200 | customer_id, name, email, country, signup_date |
| `products.csv` | 30 | product_id, product_name, category, price |
| `orders.csv` | 5,000 | order_id, customer_id, product_id, quantity, order_ts, status |
| `orders_messy.csv` | 1,050 | same as orders, but dirty (used in the capstone) |
| `events.jsonl` | 2,000 | website click-stream events with nested fields |

In [1]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [2]:
print(open(f"{DATA}/customers.csv").read()[:300])     # peek at the raw text

customer_id,name,email,country,signup_date
1,David Smith,david.smith1@outlook.com,US,2023-04-15
2,Ravi Garcia,ravi.garcia2@company.io,,2023-04-06
3,Grace Martin,grace.martin3@gmail.com,Japan,2024-10-27
4,Ravi Lopez,ravi.lopez4@yahoo.com,Germany,2023-01-07
5,Farid Lopez,farid.lopez5@outlook.com,US,20


## Reading CSV

In [3]:
raw = spark.read.csv(f"{DATA}/customers.csv")
raw.show(3)
raw.printSchema()

+-----------+-----------+--------------------+-------+-----------+
|        _c0|        _c1|                 _c2|    _c3|        _c4|
+-----------+-----------+--------------------+-------+-----------+
|customer_id|       name|               email|country|signup_date|
|          1|David Smith|david.smith1@outl...|     US| 2023-04-15|
|          2|Ravi Garcia|ravi.garcia2@comp...|   NULL| 2023-04-06|
+-----------+-----------+--------------------+-------+-----------+
only showing top 3 rows
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)



No header handling → the header became a data row, columns are named `_c0…`, and **everything is a string**. Let's fix that:

In [4]:
customers = spark.read.csv(f"{DATA}/customers.csv", header=True, inferSchema=True)
customers.show(3)
customers.printSchema()

+-----------+------------+--------------------+-------+-----------+
|customer_id|        name|               email|country|signup_date|
+-----------+------------+--------------------+-------+-----------+
|          1| David Smith|david.smith1@outl...|     US| 2023-04-15|
|          2| Ravi Garcia|ravi.garcia2@comp...|   NULL| 2023-04-06|
|          3|Grace Martin|grace.martin3@gma...|  Japan| 2024-10-27|
+-----------+------------+--------------------+-------+-----------+
only showing top 3 rows
root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



### Explicit schemas (do this in real projects)

`inferSchema=True` makes Spark read the file **an extra time** to guess types, and it can guess wrong
(e.g. product code `"00123"` becomes the integer `123`). In production, declare the schema yourself —
either as a DDL string or a `StructType`:

In [5]:
ddl = "customer_id int, name string, email string, country string, signup_date date"

struct = T.StructType([
    T.StructField("customer_id", T.IntegerType()),
    T.StructField("name", T.StringType()),
    T.StructField("email", T.StringType()),
    T.StructField("country", T.StringType()),
    T.StructField("signup_date", T.DateType()),
])

spark.read.csv(f"{DATA}/customers.csv", header=True, schema=ddl).printSchema()
spark.read.csv(f"{DATA}/customers.csv", header=True, schema=struct).printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)

root
 |-- customer_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



`spark.read.csv(path, header=True)` is a shortcut for the general pattern, which works for *every* format:
```python
spark.read.format("csv").option("header", True).schema(ddl).load(path)
```

## Reading JSON
Spark expects **JSON Lines** by default: one JSON object per line. (For a single pretty-printed JSON
document use `.option("multiLine", True)`.) Nested objects become *structs* and lists become *arrays* — lesson 08.

In [6]:
events = spark.read.json(f"{DATA}/events.jsonl")
events.printSchema()
events.show(3, truncate=False)

root
 |-- device: struct (nullable = true)
 |    |-- os: string (nullable = true)
 |    |-- version: string (nullable = true)
 |-- event: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- page: string (nullable = true)
 |    |-- referrer: string (nullable = true)
 |-- tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ts: string (nullable = true)
 |-- user_id: long (nullable = true)



+---------------+-----------+----------------------+-----------------------+-------------------+-------+
|device         |event      |properties            |tags                   |ts                 |user_id|
+---------------+-----------+----------------------+-----------------------+-------------------+-------+
|{Android, 13.5}|click      |{/product, direct}    |[email, promo]         |2024-06-22 17:09:14|107    |
|{iOS, 14.6}    |view       |{/product, newsletter}|[organic, mobile, sale]|2024-06-14 01:44:47|36     |
|{macOS, 10.4}  |add_to_cart|{/product, direct}    |[promo, email, organic]|2024-06-10 12:13:15|75     |
+---------------+-----------+----------------------+-----------------------+-------------------+-------+
only showing top 3 rows


## Handling bad records

`orders_messy.csv` has a `quantity` column containing things like `N/A`. If we declare `quantity int`,
what happens? It depends on the read **mode**:

| mode | behaviour |
|---|---|
| `PERMISSIVE` (default) | unparseable fields become `null` |
| `DROPMALFORMED` | drops the whole bad row |
| `FAILFAST` | raises an error immediately |

(We use `collect()`/`len` below rather than `count()` because Spark is clever enough to skip parsing
columns for a bare `count()` — so it wouldn't even notice the bad values!)

In [7]:
messy_schema = "order_id int, customer_id int, product_id int, quantity int, order_ts timestamp, status string"
path = f"{DATA}/orders_messy.csv"

permissive = spark.read.csv(path, header=True, schema=messy_schema)
permissive.filter(F.col("quantity").isNull()).show(3)

dropped = spark.read.csv(path, header=True, schema=messy_schema, mode="DROPMALFORMED")
print("permissive rows:", len(permissive.collect()), " dropmalformed rows:", len(dropped.collect()))

try:
    spark.read.csv(path, header=True, schema=messy_schema, mode="FAILFAST").collect()
except Exception as e:
    print("FAILFAST raised:", type(e).__name__)

+--------+-----------+----------+--------+-------------------+---------+
|order_id|customer_id|product_id|quantity|           order_ts|   status|
+--------+-----------+----------+--------+-------------------+---------+
|     622|        163|         2|    NULL|2024-05-01 17:13:51|COMPLETED|
|      87|        125|         7|    NULL|2024-05-12 17:10:34|COMPLETED|
|     492|         34|         7|    NULL|2024-03-27 07:52:32|COMPLETED|
+--------+-----------+----------+--------+-------------------+---------+
only showing top 3 rows


permissive rows: 1050  dropmalformed rows: 1024


FAILFAST raised: Py4JJavaError


## Writing data

Writing is an **action**. Spark writes a **folder**, not a single file — one `part-…` file per partition
(because each executor writes its own piece in parallel), plus a `_SUCCESS` marker.

In [8]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
orders.write.mode("overwrite").parquet(f"{OUT}/orders_parquet")
sorted(os.listdir(f"{OUT}/orders_parquet"))

['._SUCCESS.crc',
 '.part-00000-ca248f0b-e111-45f2-a027-c8275d7df349-c000.snappy.parquet.crc',
 '_SUCCESS',
 'part-00000-ca248f0b-e111-45f2-a027-c8275d7df349-c000.snappy.parquet']

### Save modes
| mode | if the output already exists… |
|---|---|
| `"error"` / `"errorifexists"` (default) | raise an error |
| `"overwrite"` | replace it |
| `"append"` | add new files alongside |
| `"ignore"` | silently do nothing |

### Why Parquet?
Parquet is the standard format for Spark/data lakes:
* **columnar** — reading 2 of 50 columns only reads those 2 from disk
* **compressed** — typically 5-10× smaller than CSV
* **typed** — the schema is stored inside the file, no inference needed
* **min/max statistics** let Spark skip chunks that can't match a filter

In [9]:
def folder_size(p):
    return sum(os.path.getsize(os.path.join(p, f)) for f in os.listdir(p))

print("CSV bytes:    ", os.path.getsize(f"{DATA}/orders.csv"))
print("Parquet bytes:", folder_size(f"{OUT}/orders_parquet"))
spark.read.parquet(f"{OUT}/orders_parquet").printSchema()    # types preserved, no inferSchema needed

CSV bytes:     213950
Parquet bytes: 87915
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_ts: timestamp (nullable = true)
 |-- status: string (nullable = true)



## Partitioned output

`partitionBy("col")` writes one sub-folder per value. Later, a query that filters on that column only
reads the matching folders (**partition pruning**) — a massive speed-up on big tables.

In [10]:
(orders
 .withColumn("month", F.month("order_ts"))
 .write.mode("overwrite")
 .partitionBy("month")
 .parquet(f"{OUT}/orders_by_month"))

print(sorted(os.listdir(f"{OUT}/orders_by_month"))[:5])
spark.read.parquet(f"{OUT}/orders_by_month").filter("month = 3").count()   # only reads month=3/

['._SUCCESS.crc', '_SUCCESS', 'month=1', 'month=10', 'month=11']


389

### Writing CSV / a single file
```python
df.write.mode("overwrite").option("header", True).csv(path)
df.coalesce(1).write...      # one partition → one file. Fine for small results, a bottleneck for big ones.
```
Other formats work the same way: `.json(...)`, `.orc(...)`, `.format("delta").save(...)` (Delta Lake, very common on Databricks).

### Exercise 1
Read `customers.csv` into `customers` using an **explicit DDL schema** where `signup_date` is a `date` (and the rest: int, string, string, string).

In [11]:
customers = spark.read.csv(f"{DATA}/customers.csv", header=True,
                           schema="customer_id int, name string, email string, country string, signup_date date")

In [12]:
# Check — run me after your answer
assert dict(customers.dtypes) == {"customer_id": "int", "name": "string", "email": "string",
                                  "country": "string", "signup_date": "date"}, customers.dtypes
assert customers.count() == 200
print("Correct!")

Correct!


### Exercise 2
Write `orders` as **Parquet** to `f"{OUT}/orders_by_status"`, **partitioned by `status`**, in overwrite mode. Then read it back into `by_status`.

In [13]:
orders.write.mode("overwrite").partitionBy("status").parquet(f"{OUT}/orders_by_status")
by_status = spark.read.parquet(f"{OUT}/orders_by_status")

In [14]:
# Check — run me after your answer
dirs = sorted(d for d in os.listdir(f"{OUT}/orders_by_status") if d.startswith("status="))
assert dirs == ["status=cancelled", "status=completed", "status=returned"], dirs
assert by_status.count() == 5000
print("Correct!")

Correct!


### Exercise 3
Using `by_status`, store the number of **completed** orders in `n_completed`.

In [15]:
n_completed = by_status.filter(F.col("status") == "completed").count()

In [16]:
# Check — run me after your answer
assert n_completed == 4035, n_completed
print("Correct!")

Correct!


## Recap
`spark.read.csv/json/parquet` · `header`, `inferSchema` vs explicit `schema` · read modes for bad data
· output is a folder of part-files · save modes · Parquet ≫ CSV · `partitionBy` for pruning.